# Review copy

MediaPipe and Sobel hand-tracker experiments, original cells 34–38. Extracted from the Drive notebook experiments_and _tests.ipynb without changing code or Markdown. Outputs and execution counts were removed to reduce file size. This part is for review and is not standalone. Original cell order is preserved.

In [ ]:
# @title
# MediaPipe 1.1.1 Tasks API detection and chirality audit.
# Does not restart the runtime or replace the installed mediapipe package.

import os
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

# --------------------------- Configuration ----------------------------------

MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        f"MediaPipe {getattr(mp, '__version__', 'unknown')} does not expose "
        "mp.tasks.vision.HandLandmarker."
    )

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# In the legacy Solutions API, HAND_CONNECTIONS defines the 21-point hand
# skeleton. Tasks API does not expose that constant, so define the same edges.
MP_HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
]

# Assert the project joint ordering has the expected MediaPipe finger chains.
# Joint 0 is wrist; then thumb, index, middle, ring, pinky, four joints each.
EXPECTED_MP_BONES = {
    tuple(sorted(edge)) for edge in MP_HAND_CONNECTIONS
}
PROJECT_BONES = {
    tuple(sorted((int(a), int(b)))) for a, b in BONE_PAIRS
}
assert EXPECTED_MP_BONES == PROJECT_BONES, (
    "KEEP_21_INDICES/BONE_PAIRS do not match MediaPipe's 21-landmark "
    "wrist-thumb-index-middle-ring-pinky ordering."
)

# ------------------------------- Helpers ------------------------------------

def make_mp_crop(gray, uv):
    """Square crop around projected GT points, enlarged to at least 256 px."""
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5

    side = max(
        float(hi[0] - lo[0]),
        float(hi[1] - lo[1]),
        16.0,
    ) * MP_CROP_MARGIN
    out_size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))

    left = center[0] - side * 0.5
    top = center[1] - side * 0.5
    scale = out_size / side

    affine = np.asarray(
        [
            [scale, 0.0, -left * scale],
            [0.0, scale, -top * scale],
        ],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        gray,
        affine,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    gt_crop_uv = uv @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def _result_hands(result, field_name):
    """Return a list for either populated or empty Tasks API result fields."""
    value = getattr(result, field_name, None)
    return [] if value is None else value


def extract_candidates(result, flipped, crop_size, gt_crop_uv):
    """Convert a Tasks result into scored, root-relative candidate poses."""
    candidates = []
    image_hands = _result_hands(result, "hand_landmarks")
    world_hands = _result_hands(result, "hand_world_landmarks")
    handedness_hands = _result_hands(result, "handedness")

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )

        # Return flipped image detections to original crop coordinates before
        # comparing them to the unflipped GT projection.
        if flipped:
            xy[:, 0] = (crop_size - 1) - xy[:, 0]

        pixel_error = np.linalg.norm(xy - gt_crop_uv, axis=-1)
        pixel_rmse = float(np.sqrt(np.mean(pixel_error ** 2)))

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )

        # MediaPipe world landmarks are meters, centered at the hand's
        # geometric center. Convert to mm and make the wrist the root.
        world_root_mm = (world_m - world_m[0:1]) * 1000.0

        # Horizontal image mirroring reverses image chirality. Undo the
        # corresponding camera-space x reflection for the flipped candidate.
        if flipped:
            world_root_mm[:, 0] *= -1.0

        handedness = None
        if hand_i < len(handedness_hands):
            try:
                handedness = handedness_hands[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                try:
                    handedness = handedness_hands[hand_i][0].display_name
                except (AttributeError, IndexError, TypeError):
                    handedness = None

        candidates.append(
            {
                "score_px": pixel_rmse,
                "pose": world_root_mm.astype(np.float32),
                "flipped": bool(flipped),
                "handedness": handedness,
            }
        )

    return candidates


def create_hand_landmarker():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(MP_MODEL_PATH)
        ),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


# ------------------------ Run validation detections -------------------------

mp_predictions = [None] * len(val_idx)
flip_choices = {"normal": 0, "flipped": 0}
flip_counts_by_segment = {
    segment: {
        "normal": 0,
        "flipped": 0,
        "detected": 0,
        "frames": 0,
    }
    for segment in sorted(val_segments)
}
match_errors_px = []
handedness_counts = {}

with create_hand_landmarker() as hand_landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        rec = records[int(dataset_i)]

        # This section expects each record to retain the source grayscale
        # image under "gray", and projected right-hand GT points under
        # "uv_image", as in the earlier Phase 0/2 loaders.
        gray = rec.get("gray")
        if gray is None:
            raise KeyError(
                "records entries need a 'gray' source-image array for the "
                "MediaPipe crop. The earlier loader must retain it."
            )

        crop, gt_crop_uv = make_mp_crop(gray, rec["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        normal_rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        normal_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=np.ascontiguousarray(normal_rgb),
        )
        normal_result = hand_landmarker.detect(normal_image)
        candidates.extend(
            extract_candidates(
                normal_result,
                flipped=False,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        flipped_crop = cv2.flip(crop, 1)
        flipped_rgb = cv2.cvtColor(flipped_crop, cv2.COLOR_GRAY2RGB)
        flipped_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=np.ascontiguousarray(flipped_rgb),
        )
        flipped_result = hand_landmarker.detect(flipped_image)
        candidates.extend(
            extract_candidates(
                flipped_result,
                flipped=True,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        segment = rec["segment"]
        if segment not in flip_counts_by_segment:
            flip_counts_by_segment[segment] = {
                "normal": 0,
                "flipped": 0,
                "detected": 0,
                "frames": 0,
            }
        flip_counts_by_segment[segment]["frames"] += 1

        if candidates:
            best = min(candidates, key=lambda item: item["score_px"])
            mp_predictions[val_pos] = best["pose"]

            choice = "flipped" if best["flipped"] else "normal"
            flip_choices[choice] += 1
            flip_counts_by_segment[segment][choice] += 1
            flip_counts_by_segment[segment]["detected"] += 1
            match_errors_px.append(best["score_px"])

            label = best["handedness"] or "unknown"
            handedness_counts[label] = handedness_counts.get(label, 0) + 1

        if (val_pos + 1) % 100 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"MediaPipe processed {val_pos + 1}/{len(val_idx)} "
                "validation frames"
            )

detected = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detection_rate = float(detected.mean()) if len(detected) else 0.0

print("\nMEDIAPIPE DETECTION / CHIRALITY AUDIT")
print(
    f"Detected: {int(detected.sum())}/{len(val_idx)} "
    f"({detection_rate:.2%})"
)
if match_errors_px:
    print(
        f"Best match chosen from normal crops: {flip_choices['normal']}; "
        f"horizontally flipped crops: {flip_choices['flipped']}; "
        f"mean 2D match RMSE: {np.mean(match_errors_px):.2f}px"
    )
else:
    print("No hand detections matched the validation frames.")
print(f"MediaPipe handedness labels: {handedness_counts}")

print("\nDETECTION RATE BY VALIDATION SEGMENT")
print(f"{'Segment':<32} {'Detected':>10} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(flip_counts_by_segment):
    row = flip_counts_by_segment[segment]
    rate = row["detected"] / max(row["frames"], 1)
    print(
        f"{str(segment):<32} {row['detected']:>10} "
        f"{row['frames']:>8} {rate:>8.2%}"
    )

In [ ]:
# @title
# MediaPipe 1.1.1 Tasks API detection and chirality audit.
# Keeps the current Colab runtime and installed MediaPipe version.

import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

# --------------------------- Configuration ----------------------------------

MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        f"MediaPipe {getattr(mp, '__version__', 'unknown')} "
        "does not expose mp.tasks.vision."
    )

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# MediaPipe landmark numbering is wrist=0, then four thumb landmarks,
# four index landmarks, four middle landmarks, four ring landmarks,
# and four pinky landmarks.
#
# HAND_CONNECTIONS is a drawing graph with palm cross-links, so it must not
# be compared directly with the project's finger-chain BONE_PAIRS.
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
PROJECT_CHAIN_BONES = {
    tuple(sorted((int(parent), int(child))))
    for parent, child in BONE_PAIRS
}
assert EXPECTED_CHAIN_BONES == PROJECT_CHAIN_BONES, (
    "BONE_PAIRS does not match the assumed MediaPipe 21-landmark "
    "wrist/thumb/index/middle/ring/pinky chain ordering."
)

# ------------------------------- Helpers ------------------------------------

def make_mp_crop(gray, uv):
    """Square crop around projected GT points, enlarged to at least 256 px."""
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5

    side = max(
        float(hi[0] - lo[0]),
        float(hi[1] - lo[1]),
        16.0,
    ) * MP_CROP_MARGIN
    out_size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))

    left = center[0] - side * 0.5
    top = center[1] - side * 0.5
    scale = out_size / side

    affine = np.asarray(
        [
            [scale, 0.0, -left * scale],
            [0.0, scale, -top * scale],
        ],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        gray,
        affine,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    gt_crop_uv = uv @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def _result_hands(result, field_name):
    value = getattr(result, field_name, None)
    return [] if value is None else value


def extract_candidates(result, flipped, crop_size, gt_crop_uv):
    """Convert Tasks results into scored, root-relative candidate poses."""
    candidates = []
    image_hands = _result_hands(result, "hand_landmarks")
    world_hands = _result_hands(result, "hand_world_landmarks")
    handedness_hands = _result_hands(result, "handedness")

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )

        # Undo image mirroring before comparing with the unflipped GT points.
        if flipped:
            xy[:, 0] = (crop_size - 1) - xy[:, 0]

        pixel_rmse = float(
            np.sqrt(np.mean(np.sum((xy - gt_crop_uv) ** 2, axis=-1)))
        )

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )

        # MediaPipe world landmarks are in meters. Convert to mm and express
        # the prediction relative to its wrist.
        world_root_mm = (world_m - world_m[0:1]) * 1000.0

        # Approximate undoing of horizontal image mirroring in camera x.
        if flipped:
            world_root_mm[:, 0] *= -1.0

        handedness = None
        if hand_i < len(handedness_hands):
            try:
                handedness = handedness_hands[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                try:
                    handedness = handedness_hands[hand_i][0].display_name
                except (AttributeError, IndexError, TypeError):
                    handedness = None

        candidates.append(
            {
                "score_px": pixel_rmse,
                "pose": world_root_mm.astype(np.float32),
                "flipped": bool(flipped),
                "handedness": handedness,
            }
        )

    return candidates


def create_hand_landmarker():
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(
            model_asset_path=str(MP_MODEL_PATH)
        ),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)


def get_record_gray(record):
    """Accept either a full grayscale frame or a stored grayscale crop."""
    gray = record.get("gray")
    if gray is None:
        gray = record.get("gray_image")
    if gray is None:
        raise KeyError(
            "Record has no source grayscale image. Store it as "
            "records[i]['gray'] in the data-loading section."
        )
    if gray.ndim != 2:
        raise ValueError("Expected a single-channel grayscale image.")
    return gray


# ------------------------ Run validation detections -------------------------

mp_predictions = [None] * len(val_idx)
flip_choices = {"normal": 0, "flipped": 0}
flip_counts_by_segment = {
    segment: {
        "normal": 0,
        "flipped": 0,
        "detected": 0,
        "frames": 0,
    }
    for segment in sorted(val_segments)
}
match_errors_px = []
handedness_counts = {}

with create_hand_landmarker() as hand_landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        rec = records[int(dataset_i)]
        gray = get_record_gray(rec)

        crop, gt_crop_uv = make_mp_crop(gray, rec["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        normal_rgb = cv2.cvtColor(crop, cv2.COLOR_GRAY2RGB)
        normal_result = hand_landmarker.detect(
            mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=np.ascontiguousarray(normal_rgb),
            )
        )
        candidates.extend(
            extract_candidates(
                normal_result,
                flipped=False,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        flipped_crop = cv2.flip(crop, 1)
        flipped_rgb = cv2.cvtColor(flipped_crop, cv2.COLOR_GRAY2RGB)
        flipped_result = hand_landmarker.detect(
            mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=np.ascontiguousarray(flipped_rgb),
            )
        )
        candidates.extend(
            extract_candidates(
                flipped_result,
                flipped=True,
                crop_size=crop_size,
                gt_crop_uv=gt_crop_uv,
            )
        )

        segment = rec["segment"]
        if segment not in flip_counts_by_segment:
            flip_counts_by_segment[segment] = {
                "normal": 0,
                "flipped": 0,
                "detected": 0,
                "frames": 0,
            }
        flip_counts_by_segment[segment]["frames"] += 1

        if candidates:
            best = min(candidates, key=lambda item: item["score_px"])
            mp_predictions[val_pos] = best["pose"]

            choice = "flipped" if best["flipped"] else "normal"
            flip_choices[choice] += 1
            flip_counts_by_segment[segment][choice] += 1
            flip_counts_by_segment[segment]["detected"] += 1
            match_errors_px.append(best["score_px"])

            label = best["handedness"] or "unknown"
            handedness_counts[label] = handedness_counts.get(label, 0) + 1

        if (val_pos + 1) % 100 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"MediaPipe processed {val_pos + 1}/{len(val_idx)} "
                "validation frames"
            )

detected = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detection_rate = float(detected.mean()) if len(detected) else 0.0

print("\nMEDIAPIPE DETECTION / CHIRALITY AUDIT")
print(
    f"Detected: {int(detected.sum())}/{len(val_idx)} "
    f"({detection_rate:.2%})"
)
if match_errors_px:
    print(
        f"Best match from normal crops: {flip_choices['normal']}; "
        f"from flipped crops: {flip_choices['flipped']}; "
        f"mean 2D match RMSE: {np.mean(match_errors_px):.2f}px"
    )
else:
    print("No hand detections matched the validation frames.")
print(f"MediaPipe handedness labels: {handedness_counts}")

print("\nDETECTION RATE BY VALIDATION SEGMENT")
print(f"{'Segment':<32} {'Detected':>10} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(flip_counts_by_segment):
    row = flip_counts_by_segment[segment]
    rate = row["detected"] / max(row["frames"], 1)
    print(
        f"{str(segment):<32} {row['detected']:>10} "
        f"{row['frames']:>8} {rate:>8.2%}"
    )

In [ ]:
# @title
# =============================================================================
# END-TO-END COLAB: COLOR MEDIAPIPE + SOBEL DIRECT REGRESSOR
# Finds/loads dataset, creates the deterministic 44/12 segment split,
# reruns MediaPipe on original color frames, trains Sobel model for 20 epochs,
# and reports results on detected frames and >=90%-detection segments.
# =============================================================================

import csv
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------- Configuration ------------------------------

SEED = 20251018
CAMERA_ID = "400485"
TRAIN_SEGMENT_COUNT = 44
VAL_SEGMENT_COUNT = 12
IMAGE_W, IMAGE_H = 128, 96
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256
TRAIN_CROP_MARGIN = 1.4
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR = 1.2e-3
WEIGHT_DECAY = 1e-4

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
FINGER_NAMES = ["Thumb", "Index", "Middle", "Ring", "Pinky"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

print(
    f"Active compute device: {DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'})"
)
print(f"MediaPipe version: {getattr(mp, '__version__', 'unknown')}")

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError(
        "This cell needs MediaPipe with mp.tasks.vision.HandLandmarker. "
        "The current runtime does not expose that API."
    )

# ------------------------------- Locate data ---------------------------------

def _manifest_candidates(root):
    if not root.exists():
        return []
    try:
        return list(root.rglob("manifest.csv"))
    except Exception:
        return []


def _choose_manifest(hits):
    if not hits:
        return None
    return next(
        (path for path in hits if "mugsy_subset" in str(path).lower()),
        hits[0],
    )


MANIFEST_PATH = None
for search_root in (DATASET_DIR, Path("/content"), Path.cwd()):
    MANIFEST_PATH = _choose_manifest(_manifest_candidates(search_root))
    if MANIFEST_PATH is not None:
        break

if MANIFEST_PATH is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped or unavailable: {exc}")
    MANIFEST_PATH = _choose_manifest(_manifest_candidates(DRIVE_ROOT))

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "Could not find manifest.csv under /content or Google Drive. "
        "Set DATASET_DIR to the extracted dataset directory and rerun."
    )

SUBSET_DIR = MANIFEST_PATH.parent


def _find_camera_file(subset_dir):
    for parent in (subset_dir, *subset_dir.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate

    search_roots = [subset_dir.parent, subset_dir.parent.parent, Path("/content")]
    if DRIVE_ROOT.exists() and str(subset_dir).startswith(str(DRIVE_ROOT)):
        search_roots.append(DRIVE_ROOT)

    for root in search_roots:
        if root.exists():
            hits = list(root.rglob("cam_params.json"))
            if hits:
                return next(
                    (p for p in hits if p.parent.name.lower() == "mugsy_cameras"),
                    hits[0],
                )
    return None


CAMERA_PATH = _find_camera_file(SUBSET_DIR)
if CAMERA_PATH is None:
    raise FileNotFoundError(
        "Could not find cam_params.json near the manifest or in /content."
    )

# Capture root is the nearest ancestor containing the camera directory, or
# otherwise the manifest's parent.
CAPTURE_DIR = SUBSET_DIR
for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
    if (
        (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
        or (parent / "keypoints_right").is_dir()
    ):
        CAPTURE_DIR = parent
        break

print(f"Manifest: {MANIFEST_PATH}")
print(f"Capture root: {CAPTURE_DIR}")
print(f"Camera: {CAMERA_PATH}")

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
rows = [
    row for row in manifest_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows found for camera {CAMERA_ID}.")

# Build one deduplicated filename index. In flattened exports, keypoints and
# images may be in keypoints_right/ and images/ rather than manifest subfolders.
print("Indexing dataset files...")
ASSET_INDEX = {}
for root in dict.fromkeys(
    [CAPTURE_DIR, SUBSET_DIR, SUBSET_DIR.parent, SUBSET_DIR.parent.parent]
):
    if not root.exists():
        continue
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {
            ".json", ".png", ".jpg", ".jpeg"
        }:
            ASSET_INDEX.setdefault(path.name.lower(), set()).add(path.resolve())


def resolve_asset(rel_path, kind):
    rel = str(rel_path).replace("\\", "/").strip()
    rel_path_obj = Path(rel)
    parts = rel_path_obj.parts
    basename = rel_path_obj.name.lower()

    # Literal and likely-root relative paths first.
    for candidate in (
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
        SUBSET_DIR.parent.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(ASSET_INDEX.get(basename, set()))

    if kind == "keypoint":
        # orig_fits/right/... must resolve to keypoints_right, never the
        # same-basename keypoints_left file.
        path_lower = [part.lower() for part in parts]
        side = "right" if "right" in path_lower else (
            "left" if "left" in path_lower else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}" in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits
    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                marker in str(p).replace("\\", "/").lower()
                for marker in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    hits = sorted(set(hits))
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        # Prefer an exact expected side folder for keypoints.
        if kind == "keypoint":
            right_hits = [
                p for p in hits
                if "keypoints_right" in str(p).replace("\\", "/").lower()
            ]
            if len(right_hits) == 1:
                return right_hits[0]
        raise FileNotFoundError(
            f"Ambiguous {kind} asset {rel!r}: "
            + ", ".join(str(p) for p in hits[:5])
        )
    raise FileNotFoundError(f"Could not resolve {kind} asset: {rel}")


with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
if CAMERA_ID not in camera_dict:
    raise KeyError(f"Camera {CAMERA_ID} is not in {CAMERA_PATH}.")
camera = camera_dict[CAMERA_ID]

CAM_R = np.asarray(camera["R"], dtype=np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], dtype=np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

# Read keypoints first, then select a plausible translation scale. Mugsy world
# keypoints are expected in mm; camera t may be stored in mm or meters.
print(f"Reading {len(rows):,} keypoint files...")
raw_world = []
keypoint_paths = []
image_paths = []
valid_rows = []

for row_i, row in enumerate(rows):
    kp_path = resolve_asset(row["orig_kp_right"], "keypoint")
    img_path = resolve_asset(row["image_path"], "image")
    with kp_path.open(encoding="utf-8") as f:
        keypoints = np.asarray(json.load(f), dtype=np.float32)
    if keypoints.ndim != 2 or keypoints.shape[1] != 3:
        raise ValueError(
            f"Expected Nx3 keypoints in {kp_path}, got {keypoints.shape}"
        )
    if max(KEEP_21_INDICES) >= len(keypoints):
        raise ValueError(
            f"KEEP_21_INDICES exceeds keypoint count {len(keypoints)} "
            f"in {kp_path}"
        )
    raw_world.append(keypoints)
    keypoint_paths.append(kp_path)
    image_paths.append(img_path)
    valid_rows.append(row)

raw_world = np.stack(raw_world).astype(np.float32)

# Translation unit audit via fraction of selected points with plausible positive
# depth and image projections inside image bounds. Select among common scales.
first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]

def _projection_in_bounds_fraction(translation_scale):
    t = CAM_T_RAW * translation_scale
    sample_n = min(len(raw_world), 256)
    selected = raw_world[:sample_n, KEEP_21_INDICES]
    cam_pts = selected @ CAM_R.T + t[None, None, :]
    z = cam_pts[..., 2]
    valid_z = z > 1.0
    z_safe = np.maximum(z, 1.0)
    u = CAM_FOCAL[0] * cam_pts[..., 0] / z_safe + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * cam_pts[..., 1] / z_safe + CAM_PRINCPT[1]
    inside = (
        valid_z
        & (u >= 0) & (u < IMAGE_ORIG_W)
        & (v >= 0) & (v < IMAGE_ORIG_H)
    )
    return float(inside.mean())


translation_candidates = [0.001, 1.0, 1000.0]
translation_scores = {
    scale: _projection_in_bounds_fraction(scale)
    for scale in translation_candidates
}
CAM_T_SCALE = max(translation_scores, key=translation_scores.get)
CAM_T = CAM_T_RAW * CAM_T_SCALE
print(
    "Camera translation scale audit: "
    + ", ".join(
        f"{scale:g}x={score:.2%}"
        for scale, score in translation_scores.items()
    )
    + f"; selected {CAM_T_SCALE:g}x"
)


def project_camera(points_cam):
    z = points_cam[..., 2]
    z_safe = np.maximum(z, 1.0)
    u = CAM_FOCAL[0] * points_cam[..., 0] / z_safe + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points_cam[..., 1] / z_safe + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


# --------------------------- Segment-wise split ------------------------------

segments = np.asarray(sorted({row["segment"] for row in valid_rows}), dtype=object)
if len(segments) != TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT:
    raise RuntimeError(
        f"Expected {TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT} segments, "
        f"found {len(segments)}."
    )

split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:TRAIN_SEGMENT_COUNT].tolist())
val_segments = set(shuffled_segments[TRAIN_SEGMENT_COUNT:].tolist())
assert not train_segments.intersection(val_segments)

train_idx = np.asarray(
    [i for i, row in enumerate(valid_rows) if row["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, row in enumerate(valid_rows) if row["segment"] in val_segments],
    dtype=np.int64,
)

# ---------------------------- Crop helpers ----------------------------------

def crop_affine(uv, image_w, image_h, margin, out_w, out_h):
    uv = np.asarray(uv, dtype=np.float32)
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        dtype=np.float32,
    )


def make_mp_crop(color_bgr, uv):
    affine = crop_affine(
        uv, IMAGE_ORIG_W, IMAGE_ORIG_H,
        MP_CROP_MARGIN, 1, 1,
    )
    # Determine square output size from the original projected box and margin.
    lo, hi = np.asarray(uv).min(axis=0), np.asarray(uv).max(axis=0)
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    center = (lo + hi) * 0.5
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    gt_crop_uv = np.asarray(uv, dtype=np.float32) @ affine[:, :2].T + affine[:, 2]
    return crop, gt_crop_uv


def make_train_crop(gray, uv):
    affine = crop_affine(
        uv, IMAGE_ORIG_W, IMAGE_ORIG_H,
        TRAIN_CROP_MARGIN, IMAGE_W, IMAGE_H,
    )
    crop = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = np.asarray(uv, dtype=np.float32) @ affine[:, :2].T + affine[:, 2]
    return crop, uv_crop


def make_sobel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


def normalize_crop_points(uv):
    uv = np.asarray(uv, dtype=np.float32)
    out = np.empty_like(uv)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out


# --------------------- MediaPipe Tasks API initialization -------------------

if not MP_MODEL_PATH.is_file():
    print(f"Downloading MediaPipe hand model to {MP_MODEL_PATH} ...")
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp_options)

# ------------------------------- Load frames --------------------------------

print(
    f"Loading {len(valid_rows):,} frames; "
    f"{len(train_idx):,} train / {len(val_idx):,} val frames across "
    f"{len(train_segments)} / {len(val_segments)} segments."
)

records = []
mp_predictions = [None] * len(val_idx)
mp_handedness = ["miss"] * len(val_idx)
mp_match_rmse = np.full(len(val_idx), np.nan, dtype=np.float32)
val_position_by_dataset_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

projection_inside_count = 0
projection_point_count = 0

try:
    for i, (row, kp_world, image_path) in enumerate(
        zip(valid_rows, raw_world, image_paths)
    ):
        color_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if color_bgr is None:
            raise FileNotFoundError(f"Could not read image {image_path}")
        if color_bgr.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
            raise ValueError(
                f"Image size changed at {image_path}: {color_bgr.shape[:2]}"
            )

        camera_pts = kp_world @ CAM_R.T + CAM_T
        selected = camera_pts[KEEP_21_INDICES]
        uv = project_camera(selected)
        root_relative = selected - selected[0:1]

        inside = (
            (selected[:, 2] > 1.0)
            & (uv[:, 0] >= 0) & (uv[:, 0] < IMAGE_ORIG_W)
            & (uv[:, 1] >= 0) & (uv[:, 1] < IMAGE_ORIG_H)
        )
        projection_inside_count += int(inside.sum())
        projection_point_count += len(inside)

        gray = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2GRAY)
        train_crop, train_uv_crop = make_train_crop(gray, uv)

        record = {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "y3d": root_relative.astype(np.float32),
            "uv_image": uv.astype(np.float32),
            "crop_gray": train_crop,
            "uv_crop": train_uv_crop.astype(np.float32),
        }
        records.append(record)

        # Run MediaPipe only on validation frames, on original color (BGR->RGB).
        val_pos = val_position_by_dataset_index.get(i)
        if val_pos is not None:
            mp_crop_bgr, gt_crop_uv = make_mp_crop(color_bgr, uv)
            mp_rgb = cv2.cvtColor(mp_crop_bgr, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(mp_rgb),
                )
            )

            image_hands = getattr(result, "hand_landmarks", None) or []
            world_hands = getattr(result, "hand_world_landmarks", None) or []
            handedness_hands = getattr(result, "handedness", None) or []
            candidates = []

            for hand_i, (image_hand, world_hand) in enumerate(
                zip(image_hands, world_hands)
            ):
                if len(image_hand) != 21 or len(world_hand) != 21:
                    continue

                pred_uv = np.asarray(
                    [
                        [lm.x * mp_rgb.shape[1], lm.y * mp_rgb.shape[0]]
                        for lm in image_hand
                    ],
                    dtype=np.float32,
                )
                match_rmse = float(
                    np.sqrt(
                        np.mean(np.sum((pred_uv - gt_crop_uv) ** 2, axis=-1))
                    )
                )

                world_m = np.asarray(
                    [[lm.x, lm.y, lm.z] for lm in world_hand],
                    dtype=np.float32,
                )
                # MediaPipe world landmarks are in meters; re-root at wrist.
                pose_mm = (world_m - world_m[0:1]) * 1000.0

                label = "unknown"
                if hand_i < len(handedness_hands):
                    try:
                        label = handedness_hands[hand_i][0].category_name
                    except (AttributeError, IndexError, TypeError):
                        pass

                candidates.append(
                    {
                        "score": match_rmse,
                        "pose": pose_mm.astype(np.float32),
                        "handedness": label,
                    }
                )

            if candidates:
                best = min(candidates, key=lambda item: item["score"])
                mp_predictions[val_pos] = best["pose"]
                mp_handedness[val_pos] = best["handedness"]
                mp_match_rmse[val_pos] = best["score"]

        if (i + 1) % 500 == 0 or i + 1 == len(valid_rows):
            print(f"Loaded and processed {i + 1}/{len(valid_rows)} frames")
finally:
    landmarker.close()

projection_fraction = (
    projection_inside_count / max(projection_point_count, 1)
)
print(
    f"Projected selected joints inside image: {projection_fraction:.2%}"
)

# Make sure the ordering matches the standard MediaPipe wrist/finger chains.
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
PROJECT_CHAIN_BONES = {
    tuple(sorted(edge)) for edge in BONE_PAIRS
}
assert EXPECTED_CHAIN_BONES == PROJECT_CHAIN_BONES, (
    "BONE_PAIRS do not match the assumed MediaPipe wrist/thumb/index/"
    "middle/ring/pinky chain ordering."
)

# -------------------------- Verify split and arrays --------------------------

assert len(records) == len(valid_rows)
assert not train_segments.intersection(val_segments)
assert len(train_segments) == TRAIN_SEGMENT_COUNT
assert len(val_segments) == VAL_SEGMENT_COUNT

y_all = np.stack([record["y3d"] for record in records]).astype(np.float32)
y2d_all = np.stack(
    [normalize_crop_points(record["uv_crop"]) for record in records]
).astype(np.float32)
gray_crops = [record["crop_gray"] for record in records]

# Mean GT bone lengths from train segments only.
canonical_lens_np = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)

# -------------------------- Sobel direct model ------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints_from_bones(bone_vectors):
    batch = bone_vectors.shape[0]
    joints = torch.zeros(
        (batch, 21, 3),
        dtype=bone_vectors.dtype,
        device=bone_vectors.device,
    )
    for idx, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + bone_vectors[:, idx]
    return joints


class SobelDirectTracker(nn.Module):
    """Phase-0 WebcamAlignedLCFTracker with a one-channel Sobel input."""

    def __init__(self, canonical_lens):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer(
            "bone_lens",
            canonical_lens.clone(),
        )

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(
            torch.flatten(self.encoder(x), 1)
        )

        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1,
            eps=1e-7,
        )
        d6 = self.latent_frame_head(feat)
        frame = rotation_6d_to_matrix(d6)

        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3),
            dtype=x.dtype,
            device=x.device,
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            phalanx_offset = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[
                :, phalanx_offset:phalanx_offset + 3
            ]

        bone_vecs = dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x = points[:, 0].copy()
    y = points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_training_sample(gray, uv_crop, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    shift_x = float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    shift_y = float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5),
        angle_deg,
        scale,
    ).astype(np.float32)
    matrix[0, 2] += shift_x
    matrix[1, 2] += shift_y

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset,
        0,
        255,
    ).astype(np.uint8)

    uv_aug = uv_crop @ matrix[:, :2].T + matrix[:, 2]
    y3d_aug = rotate_camera_xy(y3d, angle)
    uv_norm = normalize_crop_points(uv_aug)
    return make_sobel(warped), y3d_aug.astype(np.float32), uv_norm


def direct_loss(pred3d, target3d, pred2d, target2d):
    l_pos = F.l1_loss(pred3d, target3d)

    pred_bones = (
        pred3d[:, [child for _, child in BONE_PAIRS]]
        - pred3d[:, [parent for parent, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [child for _, child in BONE_PAIRS]]
        - target3d[:, [parent for parent, _ in BONE_PAIRS]]
    )

    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    l_len = F.l1_loss(pred_len, target_len)

    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    cosine = torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    l_dir = (1.0 - cosine).mean()

    l_2d = F.smooth_l1_loss(pred2d, target2d)
    return l_pos + 0.5 * l_len + 20.0 * l_dir + 15.0 * l_2d


sobel_model = SobelDirectTracker(
    torch.from_numpy(canonical_lens_np)
).to(DEVICE)

optimizer = torch.optim.AdamW(
    sobel_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=TRAIN_EPOCHS,
    eta_min=1e-4,
)
augment_rng = np.random.default_rng(SEED + 1)

print(f"\nTraining Sobel direct regressor for {TRAIN_EPOCHS} epochs...")
for epoch in range(1, TRAIN_EPOCHS + 1):
    sobel_model.train()
    order = np.random.permutation(train_idx)

    for start in range(0, len(order), BATCH_SIZE):
        batch_ids = order[start:start + BATCH_SIZE]

        samples = [
            augment_training_sample(
                gray_crops[int(i)],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                augment_rng,
            )
            for i in batch_ids
        ]

        bx = torch.from_numpy(
            np.stack([sample[0] for sample in samples])
        ).to(DEVICE)
        by3d = torch.from_numpy(
            np.stack([sample[1] for sample in samples])
        ).to(DEVICE)
        by2d = torch.from_numpy(
            np.stack([sample[2] for sample in samples])
        ).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = sobel_model(bx)
        loss = direct_loss(pred3d, by3d, pred2d, by2d)
        loss.backward()
        optimizer.step()

    scheduler.step()
    print(f"Sobel direct epoch {epoch:02d}/{TRAIN_EPOCHS} complete")

# ------------------------------ Predictions ---------------------------------

sobel_model.eval()
sobel_predictions = [None] * len(val_idx)

with torch.inference_mode():
    for val_pos, dataset_i in enumerate(val_idx):
        gray = gray_crops[int(dataset_i)]
        inp = torch.from_numpy(make_sobel(gray)[None]).to(DEVICE)
        pred3d, _, _ = sobel_model(inp)
        sobel_predictions[val_pos] = pred3d[0].cpu().numpy().astype(np.float32)

# ------------------------ Metrics and bone scaling ---------------------------

def rescale_mp_bones_to_train_mean(pose):
    pose = np.asarray(pose, dtype=np.float32)
    out = np.zeros_like(pose)

    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        length = float(np.linalg.norm(direction))
        if length < 1e-8:
            direction = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
        else:
            direction = direction / length
        out[child] = out[parent] + direction * canonical_lens_np[bone_i]

    return out


def component_metrics(pred, target):
    delta = np.asarray(pred) - np.asarray(target)
    mpjpe = float(np.linalg.norm(delta, axis=-1).mean())
    abs_xyz = np.abs(delta).mean(axis=(0, 1))
    return mpjpe, float(abs_xyz[0]), float(abs_xyz[1]), float(abs_xyz[2])


detected_mask = np.asarray(
    [pose is not None for pose in mp_predictions],
    dtype=bool,
)
detected_positions = np.flatnonzero(detected_mask)
if len(detected_positions) == 0:
    raise RuntimeError("MediaPipe detected no validation hands.")

mp_raw_predictions = [None] * len(val_idx)
mp_scaled_predictions = [None] * len(val_idx)
for pos in detected_positions:
    mp_raw_predictions[pos] = mp_predictions[pos]
    mp_scaled_predictions[pos] = rescale_mp_bones_to_train_mean(
        mp_predictions[pos]
    )

gt_val = np.stack([y_all[int(i)] for i in val_idx]).astype(np.float32)

# Detection and chirality per segment.
segment_stats = {}
for pos, dataset_i in enumerate(val_idx):
    segment = records[int(dataset_i)]["segment"]
    stats = segment_stats.setdefault(
        segment, {"frames": 0, "detected": 0}
    )
    stats["frames"] += 1
    stats["detected"] += int(detected_mask[pos])

print("\nCOLOR-FRAME MEDIAPIPE DETECTION")
print(
    f"Overall: {int(detected_mask.sum())}/{len(val_idx)} "
    f"({detected_mask.mean():.2%})"
)
print(f"{'Segment':<36} {'Detected':>9} {'Frames':>8} {'Rate':>9}")
print("-" * 66)
for segment in sorted(segment_stats, key=str):
    stats = segment_stats[segment]
    rate = stats["detected"] / max(stats["frames"], 1)
    print(
        f"{str(segment):<36} {stats['detected']:>9} "
        f"{stats['frames']:>8} {rate:>8.2%}"
    )

handedness_counts = {}
for pos in detected_positions:
    label = str(mp_handedness[pos])
    handedness_counts[label] = handedness_counts.get(label, 0) + 1
right_count = sum(
    count for label, count in handedness_counts.items()
    if label.strip().lower() == "right"
)
print("\nHANDEDNESS VS GT CHIRALITY")
print("GT chirality assumption: right hand (orig_kp_right)")
print(f"MediaPipe label counts: {handedness_counts}")
print(
    f"MediaPipe label 'Right': {right_count}/{len(detected_positions)} "
    f"({right_count / len(detected_positions):.2%})"
)

high_detection_segments = {
    segment
    for segment, stats in segment_stats.items()
    if stats["detected"] / max(stats["frames"], 1) >= 0.90
}
high_detection_positions = np.asarray(
    [
        pos for pos in detected_positions
        if records[int(val_idx[pos])]["segment"] in high_detection_segments
    ],
    dtype=np.int64,
)

def report_table(title, positions):
    print(f"\n{title}")
    print(
        f"{'Method':<25} {'MPJPE mm':>10} "
        f"{'|dx| mm':>10} {'|dy| mm':>10} {'|dz| mm':>10}"
    )
    print("-" * 70)

    methods = [
        ("MediaPipe raw", mp_raw_predictions),
        ("MediaPipe bone-scaled", mp_scaled_predictions),
        ("Sobel direct", sobel_predictions),
    ]
    for name, predictions in methods:
        pred = np.stack([predictions[int(pos)] for pos in positions])
        target = np.stack([gt_val[int(pos)] for pos in positions])
        metrics = component_metrics(pred, target)
        print(
            f"{name:<25} {metrics[0]:10.2f} "
            f"{metrics[1]:10.2f} {metrics[2]:10.2f} {metrics[3]:10.2f}"
        )


report_table("DETECTED VALIDATION FRAMES", detected_positions)

if len(high_detection_positions):
    print(
        "\nSegments meeting detection >=90%: "
        + ", ".join(map(str, sorted(high_detection_segments, key=str)))
    )
    report_table(
        "DETECTED FRAMES FROM >=90%-DETECTION SEGMENTS",
        high_detection_positions,
    )
else:
    print("\nNo validation segments reached the >=90% detection threshold.")

checkpoint_path = Path("/content/phase3_sobel_direct.pt")
torch.save(
    {
        "model_state_dict": {
            key: value.detach().cpu()
            for key, value in sobel_model.state_dict().items()
        },
        "canonical_bone_lengths_mm": canonical_lens_np,
        "seed": SEED,
        "train_segments": sorted(train_segments),
        "val_segments": sorted(val_segments),
        "camera_id": CAMERA_ID,
        "manifest_path": str(MANIFEST_PATH),
    },
    checkpoint_path,
)
print(f"\nSaved trained Sobel direct model: {checkpoint_path}")

print(
    "\nAssumptions: right-hand GT comes from orig_kp_right; world keypoints "
    "are millimeters; MediaPipe world landmarks are meters and are re-rooted "
    "at wrist joint 0; the camera translation scale is selected by in-image "
    "projection coverage; MediaPipe handedness labels are reported as returned "
    "for the unflipped color crop."
)

In [ ]:
# @title
# =============================================================================
# STANDALONE PHASE 3 AUDIT
# - Loads dataset, camera, and segment split from scratch.
# - Trains a Sobel direct regressor; no checkpoint is required.
# - Runs MediaPipe on original and mirrored color crops, keeping all candidates.
# - Reports sign, 2D, candidate, and PA-MPJPE audits.
# - Saves the trained Sobel checkpoint to Google Drive.
# =============================================================================

# Install MediaPipe if this runtime does not have it.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("mediapipe") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "mediapipe"]
    )

import csv
import itertools
import json
import os
import random
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------- Settings -----------------------------------

SEED = 20251018
CAMERA_ID = "400485"
TRAIN_SEGMENT_COUNT, VAL_SEGMENT_COUNT = 44, 12
TRAIN_EPOCHS = 20
BATCH_SIZE = 64
LR, WEIGHT_DECAY = 1.2e-3, 1e-4
IMAGE_W, IMAGE_H = 128, 96
TRAIN_CROP_MARGIN = 1.4
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
DRIVE_CHECKPOINT_PATH = DRIVE_ROOT / "phase3_sobel_direct.pt"

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]
EXPECTED_CHAIN_BONES = {
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
}
assert {tuple(sorted(edge)) for edge in BONE_PAIRS} == EXPECTED_CHAIN_BONES

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# Mount Drive now so the model can be persisted.
try:
    from google.colab import drive
    if not DRIVE_ROOT.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError("MediaPipe must expose mp.tasks.vision.HandLandmarker.")

# ---------------------------- Find dataset ---------------------------------

def manifest_hits(root):
    if not root.exists():
        return []
    try:
        return list(root.rglob("manifest.csv"))
    except Exception:
        return []


def preferred_manifest(paths):
    if not paths:
        return None
    return next(
        (p for p in paths if "mugsy_subset" in str(p).lower()),
        paths[0],
    )


manifest_path = None
for root in (DATASET_DIR, Path("/content"), Path.cwd()):
    manifest_path = preferred_manifest(manifest_hits(root))
    if manifest_path:
        break

if manifest_path is None and DRIVE_ROOT.exists():
    manifest_path = preferred_manifest(manifest_hits(DRIVE_ROOT))
if manifest_path is None:
    raise FileNotFoundError(
        "manifest.csv was not found under /content or Google Drive."
    )

subset_dir = manifest_path.parent


def find_camera_file():
    for parent in (subset_dir, *subset_dir.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate

    for root in (subset_dir.parent, subset_dir.parent.parent, Path("/content")):
        if root.exists():
            found = list(root.rglob("cam_params.json"))
            if found:
                return next(
                    (p for p in found if p.parent.name.lower() == "mugsy_cameras"),
                    found[0],
                )
    if DRIVE_ROOT.exists():
        found = list(DRIVE_ROOT.rglob("cam_params.json"))
        if found:
            return found[0]
    return None


camera_path = find_camera_file()
if camera_path is None:
    raise FileNotFoundError("cam_params.json was not found.")

capture_dir = subset_dir
for parent in (subset_dir, *subset_dir.parents):
    if (
        (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
        or (parent / "keypoints_right").is_dir()
    ):
        capture_dir = parent
        break

with manifest_path.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))
rows = [
    row for row in all_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows for camera {CAMERA_ID}.")

# Index assets, deduplicating paths found through overlapping roots.
asset_index = {}
roots = list(dict.fromkeys(
    [capture_dir, subset_dir, subset_dir.parent, subset_dir.parent.parent]
))
for root in roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in {
                ".json", ".png", ".jpg", ".jpeg"
            }:
                asset_index.setdefault(path.name.lower(), set()).add(path.resolve())


def resolve_asset(relative_path, kind):
    rel = str(relative_path).replace("\\", "/").strip()
    rel_path = Path(rel)
    for candidate in (
        Path(rel),
        capture_dir / rel,
        subset_dir / rel,
        subset_dir.parent / rel,
        subset_dir.parent.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(asset_index.get(rel_path.name.lower(), set()))
    if kind == "keypoint":
        parts = [p.lower() for p in rel_path.parts]
        side = "right" if "right" in parts else (
            "left" if "left" in parts else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}"
                in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits
    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                marker in str(p).replace("\\", "/").lower()
                for marker in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    hits = sorted(set(hits))
    if len(hits) == 1:
        return hits[0]
    if kind == "keypoint":
        right_hits = [
            p for p in hits
            if "keypoints_right" in str(p).replace("\\", "/").lower()
        ]
        if len(right_hits) == 1:
            return right_hits[0]
    raise FileNotFoundError(
        f"Cannot uniquely resolve {kind} asset {rel!r}; matches={hits[:5]}"
    )


# ---------------------------- Camera and split -----------------------------

with camera_path.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
camera = camera_dict[CAMERA_ID]
CAM_R = np.asarray(camera["R"], np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], np.float32).reshape(2)


def project(points):
    z = np.maximum(points[..., 2], 1.0)
    u = CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


segments = np.asarray(sorted({r["segment"] for r in rows}), dtype=object)
if len(segments) != TRAIN_SEGMENT_COUNT + VAL_SEGMENT_COUNT:
    raise RuntimeError(
        f"Expected 56 segments (44 train, 12 val); found {len(segments)}."
    )
split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:TRAIN_SEGMENT_COUNT].tolist())
val_segments = set(shuffled_segments[TRAIN_SEGMENT_COUNT:].tolist())
train_idx = np.asarray(
    [i for i, r in enumerate(rows) if r["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, r in enumerate(rows) if r["segment"] in val_segments],
    dtype=np.int64,
)

# Read labels and resolve images.
world_points = []
image_paths = []
for row in rows:
    kp_path = resolve_asset(row["orig_kp_right"], "keypoint")
    image_path = resolve_asset(row["image_path"], "image")
    with kp_path.open(encoding="utf-8") as f:
        points = np.asarray(json.load(f), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError(f"Expected Nx3 keypoints in {kp_path}.")
    if max(KEEP_21_INDICES) >= len(points):
        raise ValueError(f"Too few keypoints in {kp_path}.")
    world_points.append(points)
    image_paths.append(image_path)
world_points = np.stack(world_points)

first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]


def camera_in_bounds_fraction(t_scale):
    points = world_points[:min(256, len(world_points)), KEEP_21_INDICES]
    cam = points @ CAM_R.T + CAM_T_RAW * t_scale
    uv = project(cam)
    inside = (
        (cam[..., 2] > 1.0)
        & (uv[..., 0] >= 0) & (uv[..., 0] < IMAGE_ORIG_W)
        & (uv[..., 1] >= 0) & (uv[..., 1] < IMAGE_ORIG_H)
    )
    return float(inside.mean())


T_SCALE = max((0.001, 1.0, 1000.0), key=camera_in_bounds_fraction)
CAM_T = CAM_T_RAW * T_SCALE

# ------------------------------- Crops --------------------------------------

def crop_affine(uv, out_w, out_h, margin):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        np.float32,
    )


def make_mp_crop(color_bgr, uv):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0)
    side *= MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop, uv @ affine[:, :2].T + affine[:, 2]


def make_sobel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


def normalize_uv(uv):
    out = np.empty_like(uv, dtype=np.float32)
    out[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return out


# ---------------------------- Stage the dataset ----------------------------

records = []
for i, (row, world, image_path) in enumerate(
    zip(rows, world_points, image_paths)
):
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(image_path)
    if color.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
        raise ValueError(f"Image dimensions differ at {image_path}.")

    cam = world @ CAM_R.T + CAM_T
    selected = cam[KEEP_21_INDICES]
    uv = project(selected)
    y3d = selected - selected[0:1]

    gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    affine = crop_affine(uv, IMAGE_W, IMAGE_H, TRAIN_CROP_MARGIN)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    uv_crop = uv @ affine[:, :2].T + affine[:, 2]

    records.append(
        {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "image_path": image_path,
            "uv_image": uv.astype(np.float32),
            "uv_crop": uv_crop.astype(np.float32),
            "crop_gray": crop_gray,
            "y3d": y3d.astype(np.float32),
        }
    )

y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
canonical_lens_np = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)

# -------------------------- Train Sobel model -------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints_from_bones(vectors):
    joints = torch.zeros(
        (vectors.shape[0], 21, 3), dtype=vectors.dtype, device=vectors.device
    )
    for k, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + vectors[:, k]
    return joints


class SobelDirectTracker(nn.Module):
    def __init__(self, lengths):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", lengths.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1, eps=1e-7,
        )
        frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1, eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3), dtype=x.dtype, device=x.device
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            start = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[:, start:start + 3]

        joints = reconstruct_joints_from_bones(
            dirs * self.bone_lens.view(1, 20, 1)
        )
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)
    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    return make_sobel(warped), rotate_xy(y3d, angle), normalize_uv(uv_aug)


def direct_loss(pred3d, target3d, pred2d, target2d):
    pos = F.l1_loss(pred3d, target3d)
    pred_bones = (
        pred3d[:, [c for _, c in BONE_PAIRS]]
        - pred3d[:, [p for p, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [c for _, c in BONE_PAIRS]]
        - target3d[:, [p for p, _ in BONE_PAIRS]]
    )
    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    length = F.l1_loss(pred_len, target_len)
    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction = (
        1.0 - torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    ).mean()
    reprojection = F.smooth_l1_loss(pred2d, target2d)
    return pos + 0.5 * length + 20.0 * direction + 15.0 * reprojection


sobel_model = SobelDirectTracker(
    torch.from_numpy(canonical_lens_np)
).to(DEVICE)
optimizer = torch.optim.AdamW(
    sobel_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4
)
augment_rng = np.random.default_rng(SEED + 1)

for epoch in range(TRAIN_EPOCHS):
    sobel_model.train()
    order = np.random.permutation(train_idx)
    for start in range(0, len(order), BATCH_SIZE):
        batch_ids = order[start:start + BATCH_SIZE]
        samples = [
            augment(
                records[int(i)]["crop_gray"],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                augment_rng,
            )
            for i in batch_ids
        ]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        by3 = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        by2 = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = sobel_model(bx)
        loss = direct_loss(pred3d, by3, pred2d, by2)
        loss.backward()
        optimizer.step()
    scheduler.step()

# Persist for future Colab sessions.
checkpoint_payload = {
    "model_state_dict": {
        k: v.detach().cpu() for k, v in sobel_model.state_dict().items()
    },
    "canonical_bone_lengths_mm": canonical_lens_np,
    "seed": SEED,
    "train_segments": sorted(train_segments),
    "val_segments": sorted(val_segments),
    "camera_id": CAMERA_ID,
    "manifest_path": str(manifest_path),
}
if DRIVE_ROOT.exists():
    DRIVE_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint_payload, DRIVE_CHECKPOINT_PATH)
    checkpoint_saved_to = DRIVE_CHECKPOINT_PATH
else:
    checkpoint_saved_to = Path("/content/phase3_sobel_direct.pt")
    torch.save(checkpoint_payload, checkpoint_saved_to)

sobel_model.eval()

# ---------------------- MediaPipe model and candidates ----------------------

if not MP_MODEL_PATH.is_file():
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)


def extract_candidates(result, crop_size, gt_crop_uv, mirrored):
    candidates = []
    image_hands = getattr(result, "hand_landmarks", None) or []
    world_hands = getattr(result, "hand_world_landmarks", None) or []
    handedness = getattr(result, "handedness", None) or []

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue
        xy = np.asarray(
            [[lm.x * crop_size, lm.y * crop_size] for lm in image_hand],
            dtype=np.float32,
        )
        if mirrored:
            xy[:, 0] = crop_size - 1 - xy[:, 0]
        pixel_error = float(np.linalg.norm(xy - gt_crop_uv, axis=-1).mean())

        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )
        pose = (world_m - world_m[0:1]) * 1000.0
        if mirrored:
            pose[:, 0] *= -1.0

        label = "unknown"
        if hand_i < len(handedness):
            try:
                label = handedness[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                pass
        candidates.append(
            {
                "pose": pose.astype(np.float32),
                "xy": xy,
                "pixel_error": pixel_error,
                "mirrored_input": bool(mirrored),
                "hand_index": hand_i,
                "handedness": label,
            }
        )
    return candidates


all_candidates = [[] for _ in val_idx]
normal_2d_errors, best_2d_errors = [], []

with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        record = records[int(dataset_i)]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        crop_bgr, gt_crop_uv = make_mp_crop(color, record["uv_image"])
        crop_size = crop_bgr.shape[0]
        found = []

        for mirrored in (False, True):
            view_bgr = cv2.flip(crop_bgr, 1) if mirrored else crop_bgr
            view_rgb = cv2.cvtColor(view_bgr, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(view_rgb),
                )
            )
            found.extend(
                extract_candidates(result, crop_size, gt_crop_uv, mirrored)
            )

        all_candidates[val_pos] = found
        normal = [c for c in found if not c["mirrored_input"]]
        if normal:
            normal_2d_errors.append(
                min(normal, key=lambda c: c["pixel_error"])["pixel_error"]
            )
        if found:
            best_2d_errors.append(
                min(found, key=lambda c: c["pixel_error"])["pixel_error"]
            )

detected_positions = np.asarray(
    [i for i, c in enumerate(all_candidates) if c],
    dtype=np.int64,
)
if not len(detected_positions):
    raise RuntimeError("MediaPipe detected no validation hands.")

nearest_candidates = [
    min(c, key=lambda candidate: candidate["pixel_error"]) if c else None
    for c in all_candidates
]

# Sobel inference on validation crops.
sobel_predictions = [None] * len(val_idx)
with torch.inference_mode():
    for pos, dataset_i in enumerate(val_idx):
        x = torch.from_numpy(
            make_sobel(records[int(dataset_i)]["crop_gray"])[None]
        ).to(DEVICE)
        joints, _, _ = sobel_model(x)
        sobel_predictions[pos] = joints[0].cpu().numpy().astype(np.float32)

gt_val = [records[int(i)]["y3d"] for i in val_idx]


def mpjpe(pred, target):
    return float(np.linalg.norm(pred - target, axis=-1).mean())


def rescale_bones(pose):
    result = np.zeros_like(pose, dtype=np.float32)
    lengths = canonical_lens_np
    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        norm = float(np.linalg.norm(direction))
        direction = (
            np.array([1.0, 0.0, 0.0], np.float32)
            if norm < 1e-8 else direction / norm
        )
        result[child] = result[parent] + direction * lengths[bone_i]
    return result


def pa_mpjpe_one(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    x = pred - pred.mean(axis=0, keepdims=True)
    y = target - target.mean(axis=0, keepdims=True)
    u, singular, vt = np.linalg.svd(x.T @ y)
    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(u @ vt)
    rotation = u @ correction @ vt
    denom = np.sum(x * x)
    if denom < 1e-12:
        return float("nan")
    scale = np.sum(singular * np.diag(correction)) / denom
    aligned = scale * (x @ rotation) + target.mean(axis=0, keepdims=True)
    return float(np.linalg.norm(aligned - target, axis=-1).mean())


mp_raw = [None] * len(val_idx)
mp_scaled = [None] * len(val_idx)
for pos in detected_positions:
    pose = nearest_candidates[int(pos)]["pose"]
    mp_raw[int(pos)] = pose
    mp_scaled[int(pos)] = rescale_bones(pose)

# ------------------------------- Tables -------------------------------------

print("FRAME AUDIT — 2D-NEAREST CANDIDATE")
print(f"{'XYZ signs':<12} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 34)
for signs in itertools.product((1.0, -1.0), repeat=3):
    errors = [
        mpjpe(
            nearest_candidates[int(pos)]["pose"]
            * np.asarray(signs, np.float32),
            gt_val[int(pos)],
        )
        for pos in detected_positions
    ]
    label = "".join("+" if s > 0 else "-" for s in signs)
    print(f"{label:<12} {len(errors):>8} {np.mean(errors):10.2f}")

print("\n2D CHECK — MEAN JOINT DISTANCE")
print(f"{'Candidate pool':<36} {'Frames':>8} {'Error px':>10}")
print("-" * 58)
print(
    f"{'Original crop only':<36} {len(normal_2d_errors):>8} "
    f"{np.mean(normal_2d_errors):10.2f}"
)
print(
    f"{'Best original/mirrored per frame':<36} {len(best_2d_errors):>8} "
    f"{np.mean(best_2d_errors):10.2f}"
)

nearest_errors, oracle_errors = [], []
different = 0
for pos in detected_positions:
    pos = int(pos)
    candidates = all_candidates[pos]
    nearest = min(candidates, key=lambda c: c["pixel_error"])
    oracle = min(candidates, key=lambda c: mpjpe(c["pose"], gt_val[pos]))
    nearest_errors.append(mpjpe(nearest["pose"], gt_val[pos]))
    oracle_errors.append(mpjpe(oracle["pose"], gt_val[pos]))
    different += int(
        (nearest["mirrored_input"], nearest["hand_index"])
        != (oracle["mirrored_input"], oracle["hand_index"])
    )

print("\nCANDIDATE CHECK")
print(f"{'Selection':<28} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 50)
print(f"{'2D-nearest':<28} {len(nearest_errors):>8} {np.mean(nearest_errors):10.2f}")
print(f"{'3D oracle':<28} {len(oracle_errors):>8} {np.mean(oracle_errors):10.2f}")
print(
    f"Different candidate identity: {different}/{len(detected_positions)} "
    f"({different / len(detected_positions):.2%})"
)

print("\nPA-MPJPE — PER-FRAME ROTATION + SCALE")
print(f"{'Method':<26} {'Frames':>8} {'PA-MPJPE mm':>13}")
print("-" * 52)
for name, predictions in (
    ("MediaPipe raw", mp_raw),
    ("MediaPipe bone-scaled", mp_scaled),
    ("Sobel direct", sobel_predictions),
):
    values = [
        pa_mpjpe_one(predictions[int(pos)], gt_val[int(pos)])
        for pos in detected_positions
    ]
    print(f"{name:<26} {len(values):>8} {np.nanmean(values):13.2f}")

# -------------------------------- Overlays -----------------------------------

def draw_overlay(candidate, crop_bgr, gt_uv, path):
    canvas = crop_bgr.copy()
    gt_points = np.rint(gt_uv).astype(int)
    mp_points = np.rint(candidate["xy"]).astype(int)
    for parent, child in BONE_PAIRS:
        cv2.line(canvas, tuple(gt_points[parent]), tuple(gt_points[child]),
                 (0, 255, 0), 2, cv2.LINE_AA)
        cv2.line(canvas, tuple(mp_points[parent]), tuple(mp_points[child]),
                 (0, 0, 255), 2, cv2.LINE_AA)
    for point in gt_points:
        cv2.circle(canvas, tuple(point), 4, (0, 255, 0), -1, cv2.LINE_AA)
    for point in mp_points:
        cv2.circle(canvas, tuple(point), 4, (255, 0, 0), -1, cv2.LINE_AA)
    if not cv2.imwrite(str(path), canvas):
        raise IOError(f"Could not save {path}")


overlay_dir = Path("/content/phase3_audit_overlays")
overlay_dir.mkdir(parents=True, exist_ok=True)
saved = []

for segment_name in ("T0011", "T0020"):
    positions = [
        int(pos)
        for pos, dataset_i in enumerate(val_idx)
        if str(records[int(dataset_i)]["segment"]) == segment_name
        and all_candidates[int(pos)]
    ]
    if len(positions) < 2:
        raise RuntimeError(
            f"{segment_name} needs two detected validation frames; "
            f"found {len(positions)}."
        )

    for rank, pos in enumerate(positions[:2], start=1):
        dataset_i = int(val_idx[pos])
        record = records[dataset_i]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        crop, gt_uv = make_mp_crop(color, record["uv_image"])
        frame_id = str(record.get("frame_id", pos))
        safe_id = "".join(
            ch if ch.isalnum() or ch in "-_" else "_" for ch in frame_id
        )
        path = overlay_dir / f"{segment_name}_{rank}_{safe_id}.png"
        draw_overlay(nearest_candidates[pos], crop, gt_uv, path)
        saved.append(path)

print("\nOVERLAYS")
for path in saved:
    print(path)

print(f"\nSobel checkpoint saved to Google Drive: {DRIVE_CHECKPOINT_PATH}")
print(
    "\nAssumptions: right-hand GT is orig_kp_right; 3D keypoints are mm; "
    "MediaPipe world landmarks are meters and are wrist-rooted. Mirrored "
    "candidate x is reflected back before scoring. The 3D oracle uses GT and "
    "is an upper-bound diagnostic; PA-MPJPE fits proper rotation and uniform "
    "scale per frame. Camera translation units are selected by image coverage."
)

In [ ]:
# =============================================================================
# STANDALONE COLAB RUNNER — PHASE 3 AUDIT
# - Searches only the dataset/capture directory, never / or all of /content.
# - Loads the right-hand data, camera and deterministic 44/12 segment split.
# - Trains the Sobel direct regressor, with per-epoch resumable state on Drive.
# - Runs MediaPipe on normal and mirrored ORIGINAL COLOR crops.
# - Keeps every MediaPipe candidate, creates audits and four overlays.
# - Reports RAM/VRAM and emits regular progress output to avoid idle timeouts.
# =============================================================================

# Install MediaPipe in this runtime if it is absent.
import importlib.util
import subprocess
import sys
import os
import time
import csv
import json
import random
import itertools
import urllib.request
from pathlib import Path

if importlib.util.find_spec("mediapipe") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "mediapipe"]
    )

import cv2
import mediapipe as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------- Settings ----------------------------------

SEED = 20251018
CAMERA_ID = "400485"
N_TRAIN_SEGMENTS = 44
N_VAL_SEGMENTS = 12
N_EPOCHS = 20
LR = 1.2e-3
WEIGHT_DECAY = 1e-4
IMAGE_W, IMAGE_H = 128, 96
TRAIN_CROP_MARGIN = 1.4
MP_CROP_MARGIN = 1.6
MP_CROP_MIN_SIZE = 256

DATASET_DIR = Path("/content/dataset")
DRIVE_ROOT = Path("/content/drive/MyDrive")
MP_MODEL_PATH = Path("/content/hand_landmarker.task")
MP_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
RESUME_PATH = DRIVE_ROOT / "phase3_sobel_direct_resume.pt"
FINAL_MODEL_PATH = DRIVE_ROOT / "phase3_sobel_direct.pt"
OVERLAY_DIR = Path("/content/phase3_audit_overlays")

KEEP_21_INDICES = [
    0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22
]
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (0, 9), (9, 10), (10, 11), (11, 12),
    (0, 13), (13, 14), (14, 15), (15, 16),
    (0, 17), (17, 18), (18, 19), (19, 20),
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# Use smaller batches on devices with less free VRAM.
if DEVICE.type == "cuda":
    free_vram_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
    BATCH_SIZE = 64 if free_vram_gb >= 8 else 32 if free_vram_gb >= 4 else 16
else:
    BATCH_SIZE = 16

# -------------------------------- Resources ---------------------------------

try:
    import psutil
except ImportError:
    psutil = None


def memory_status(stage):
    ram = "RAM unavailable"
    if psutil is not None:
        proc = psutil.Process(os.getpid())
        vm = psutil.virtual_memory()
        ram = (
            f"process={proc.memory_info().rss / (1024**3):.2f} GiB, "
            f"available={vm.available / (1024**3):.2f} GiB"
        )

    if DEVICE.type == "cuda":
        free_b, total_b = torch.cuda.mem_get_info()
        gpu = (
            f"{torch.cuda.get_device_name(0)}, "
            f"allocated={torch.cuda.memory_allocated() / (1024**3):.2f} GiB, "
            f"reserved={torch.cuda.memory_reserved() / (1024**3):.2f} GiB, "
            f"free={free_b / (1024**3):.2f}/{total_b / (1024**3):.2f} GiB"
        )
    else:
        gpu = f"CPU, {torch.get_num_threads()} threads"

    print(f"[{stage}] {gpu}; RAM {ram}", flush=True)


print(
    f"Device: {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""),
    flush=True,
)
memory_status("start")

# ---------------------------- Google Drive ---------------------------------

try:
    from google.colab import drive
    if not DRIVE_ROOT.exists():
        print("Mounting Google Drive for resumable model saves...", flush=True)
        drive.mount("/content/drive")
except Exception as exc:
    print(f"Drive mount unavailable: {exc}", flush=True)

DRIVE_AVAILABLE = DRIVE_ROOT.exists()
if not hasattr(mp, "tasks") or not hasattr(mp.tasks, "vision"):
    raise RuntimeError("MediaPipe mp.tasks.vision.HandLandmarker is required.")

if not MP_MODEL_PATH.exists():
    print("Downloading MediaPipe hand model...", flush=True)
    urllib.request.urlretrieve(MP_MODEL_URL, MP_MODEL_PATH)

# ----------------------- Bounded manifest discovery ------------------------

# Check the known dataset path first. Avoid rglob('/content') or rglob('/').
def find_manifest():
    direct = DATASET_DIR / "manifest.csv"
    if direct.is_file():
        return direct

    if DATASET_DIR.exists():
        print(f"Searching only inside {DATASET_DIR} for manifest.csv...", flush=True)
        matches = list(DATASET_DIR.rglob("manifest.csv"))
        if matches:
            return next(
                (p for p in matches if "mugsy_subset" in str(p).lower()),
                matches[0],
            )

    if DRIVE_AVAILABLE:
        # Drive search occurs only if the local dataset path had no manifest.
        print(f"Searching Google Drive for manifest.csv...", flush=True)
        matches = list(DRIVE_ROOT.rglob("manifest.csv"))
        if matches:
            return next(
                (p for p in matches if "mugsy_subset" in str(p).lower()),
                matches[0],
            )
    return None


MANIFEST_PATH = find_manifest()
if MANIFEST_PATH is None:
    raise FileNotFoundError(
        "manifest.csv not found. Put the dataset under /content/dataset "
        "or mount Drive and set DATASET_DIR to its extracted folder."
    )

SUBSET_DIR = MANIFEST_PATH.parent


def find_camera_path():
    # Walk upward only from the manifest. No broad recursive camera search.
    for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
        for candidate in (
            parent / "Mugsy_cameras" / "cam_params.json",
            parent / "cam_params.json",
        ):
            if candidate.is_file():
                return candidate
        # Do not climb above /content or a mounted Drive root.
        if parent == Path("/content") or parent == DRIVE_ROOT:
            break
    return None


CAMERA_PATH = find_camera_path()
if CAMERA_PATH is None:
    raise FileNotFoundError(
        "cam_params.json was not found beside the manifest or in its "
        "Mugsy_cameras directory."
    )

CAPTURE_DIR = SUBSET_DIR
for parent in (SUBSET_DIR, *SUBSET_DIR.parents):
    if (
        (parent / "keypoints_right").is_dir()
        or (parent / "Mugsy_cameras" / "cam_params.json").is_file()
        or (parent / "cam_params.json").is_file()
    ):
        CAPTURE_DIR = parent
        break
    if parent == Path("/content") or parent == DRIVE_ROOT:
        break

print(f"Manifest: {MANIFEST_PATH}", flush=True)
print(f"Capture directory: {CAPTURE_DIR}", flush=True)
print(f"Camera file: {CAMERA_PATH}", flush=True)

# --------------------- Index only the capture directory ---------------------

# The earlier timeout risk was indexing CAPTURE_DIR plus its ancestors, which
# could make a parent path equal to / and scan the entire runtime.
ASSET_INDEX = {}
allowed_suffixes = {".json", ".png", ".jpg", ".jpeg"}
print(f"Indexing assets under {CAPTURE_DIR} only...", flush=True)

for count, path in enumerate(CAPTURE_DIR.rglob("*"), start=1):
    if path.is_file() and path.suffix.lower() in allowed_suffixes:
        ASSET_INDEX.setdefault(path.name.lower(), []).append(path.resolve())
    if count % 5000 == 0:
        print(f"  scanned {count:,} paths", flush=True)

print(
    f"Indexed {sum(len(v) for v in ASSET_INDEX.values()):,} paths.",
    flush=True,
)


def resolve_asset(relative_path, kind):
    rel = str(relative_path).replace("\\", "/").strip()
    rel_path = Path(rel)

    # Fast direct paths first.
    for candidate in (
        Path(rel),
        CAPTURE_DIR / rel,
        SUBSET_DIR / rel,
        SUBSET_DIR.parent / rel,
    ):
        if candidate.is_file():
            return candidate.resolve()

    hits = list(dict.fromkeys(ASSET_INDEX.get(rel_path.name.lower(), [])))

    if kind == "keypoint":
        parts = [p.lower() for p in rel_path.parts]
        side = "right" if "right" in parts else (
            "left" if "left" in parts else None
        )
        if side:
            side_hits = [
                p for p in hits
                if f"keypoints_{side}"
                in str(p).replace("\\", "/").lower()
            ]
            if side_hits:
                hits = side_hits

    elif kind == "image":
        image_hits = [
            p for p in hits
            if any(
                token in str(p).replace("\\", "/").lower()
                for token in ("/images/", "/image/", "/rgb/")
            )
        ]
        if image_hits:
            hits = image_hits

    if len(hits) == 1:
        return hits[0]

    if kind == "keypoint":
        right_hits = [
            p for p in hits
            if "keypoints_right" in str(p).replace("\\", "/").lower()
        ]
        if len(right_hits) == 1:
            return right_hits[0]

    raise FileNotFoundError(
        f"Could not uniquely resolve {kind} asset {rel!r}; "
        f"matches={[str(p) for p in hits[:5]]}"
    )

# ----------------------------- Read metadata --------------------------------

with MANIFEST_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))
rows = [
    row for row in all_rows
    if row.get("camera_id", CAMERA_ID) == CAMERA_ID
]
if not rows:
    raise RuntimeError(f"No manifest rows for camera {CAMERA_ID}.")

with CAMERA_PATH.open(encoding="utf-8") as f:
    camera_dict = json.load(f)
if CAMERA_ID not in camera_dict:
    raise KeyError(f"Camera {CAMERA_ID} absent from {CAMERA_PATH}.")

camera = camera_dict[CAMERA_ID]
CAM_R = np.asarray(camera["R"], dtype=np.float32).reshape(3, 3)
CAM_T_RAW = np.asarray(camera["t"], dtype=np.float32).reshape(3)
CAM_FOCAL = np.asarray(camera["focal"], dtype=np.float32).reshape(2)
CAM_PRINCPT = np.asarray(camera["princpt"], dtype=np.float32).reshape(2)

segments = np.asarray(sorted({row["segment"] for row in rows}), dtype=object)
if len(segments) != 56:
    raise RuntimeError(f"Expected 56 segments; found {len(segments)}.")

split_rng = np.random.default_rng(SEED)
shuffled_segments = segments.copy()
split_rng.shuffle(shuffled_segments)
train_segments = set(shuffled_segments[:44].tolist())
val_segments = set(shuffled_segments[44:].tolist())
train_idx = np.asarray(
    [i for i, row in enumerate(rows) if row["segment"] in train_segments],
    dtype=np.int64,
)
val_idx = np.asarray(
    [i for i, row in enumerate(rows) if row["segment"] in val_segments],
    dtype=np.int64,
)

# Resolve all paths once, then load the relatively small keypoint JSONs.
print(f"Resolving {len(rows):,} manifest assets...", flush=True)
keypoint_paths, image_paths = [], []
for i, row in enumerate(rows):
    keypoint_paths.append(resolve_asset(row["orig_kp_right"], "keypoint"))
    image_paths.append(resolve_asset(row["image_path"], "image"))
    if (i + 1) % 1000 == 0:
        print(f"  resolved {i + 1:,}/{len(rows):,}", flush=True)

world_points = []
for i, path in enumerate(keypoint_paths):
    with path.open(encoding="utf-8") as f:
        points = np.asarray(json.load(f), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError(f"Expected Nx3 points in {path}; got {points.shape}.")
    if len(points) <= max(KEEP_21_INDICES):
        raise ValueError(f"Too few keypoints in {path}.")
    world_points.append(points)
    if (i + 1) % 1000 == 0:
        print(f"  loaded keypoints {i + 1:,}/{len(rows):,}", flush=True)
world_points = np.stack(world_points).astype(np.float32)

first_image = cv2.imread(str(image_paths[0]), cv2.IMREAD_COLOR)
if first_image is None:
    raise FileNotFoundError(image_paths[0])
IMAGE_ORIG_H, IMAGE_ORIG_W = first_image.shape[:2]


def project(points):
    z = np.maximum(points[..., 2], 1.0)
    u = CAM_FOCAL[0] * points[..., 0] / z + CAM_PRINCPT[0]
    v = CAM_FOCAL[1] * points[..., 1] / z + CAM_PRINCPT[1]
    return np.stack((u, v), axis=-1).astype(np.float32)


def in_frame_fraction(t_scale):
    sample = world_points[:min(len(world_points), 256), KEEP_21_INDICES]
    camera_sample = sample @ CAM_R.T + CAM_T_RAW * t_scale
    uv = project(camera_sample)
    inside = (
        (camera_sample[..., 2] > 1.0)
        & (uv[..., 0] >= 0) & (uv[..., 0] < IMAGE_ORIG_W)
        & (uv[..., 1] >= 0) & (uv[..., 1] < IMAGE_ORIG_H)
    )
    return float(inside.mean())


T_SCALE_OPTIONS = (0.001, 1.0, 1000.0)
CAM_T_SCALE = max(T_SCALE_OPTIONS, key=in_frame_fraction)
CAM_T = CAM_T_RAW * CAM_T_SCALE
print(
    f"Camera translation scale: {CAM_T_SCALE:g}; "
    f"projection in-frame fraction={in_frame_fraction(CAM_T_SCALE):.2%}",
    flush=True,
)

# -------------------------- Crop and Sobel helpers --------------------------

def affine_for_crop(uv, out_w, out_h, margin):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0) * margin
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    return np.asarray(
        [
            [out_w / side, 0.0, -left * out_w / side],
            [0.0, out_h / side, -top * out_h / side],
        ],
        dtype=np.float32,
    )


def make_mp_crop(color_bgr, uv):
    lo, hi = uv.min(axis=0), uv.max(axis=0)
    center = (lo + hi) * 0.5
    side = max(float(hi[0] - lo[0]), float(hi[1] - lo[1]), 16.0)
    side *= MP_CROP_MARGIN
    size = max(MP_CROP_MIN_SIZE, int(np.ceil(side)))
    left, top = center[0] - side * 0.5, center[1] - side * 0.5
    scale = size / side
    affine = np.asarray(
        [[scale, 0.0, -left * scale], [0.0, scale, -top * scale]],
        dtype=np.float32,
    )
    crop = cv2.warpAffine(
        color_bgr, affine, (size, size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return crop, uv @ affine[:, :2].T + affine[:, 2]


def normalize_uv(uv):
    result = np.empty_like(uv, dtype=np.float32)
    result[:, 0] = 2.0 * uv[:, 0] / IMAGE_W - 1.0
    result[:, 1] = 2.0 * uv[:, 1] / IMAGE_H - 1.0
    return result


def sobel_channel(gray_u8):
    gray = gray_u8.astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3, scale=1.0 / 1020.0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3, scale=1.0 / 1020.0)
    mag = np.clip(np.sqrt(gx * gx + gy * gy), 0.0, 1.0)
    return mag[None].astype(np.float32)


# --------------------------- Stage small crops ------------------------------

print("Staging grayscale crops and labels (full color frames are not cached)...",
      flush=True)
records = []
for i, (row, world, image_path) in enumerate(
    zip(rows, world_points, image_paths)
):
    color = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(image_path)
    if color.shape[:2] != (IMAGE_ORIG_H, IMAGE_ORIG_W):
        raise ValueError(f"Unexpected image dimensions at {image_path}.")

    camera_points = world @ CAM_R.T + CAM_T
    selected = camera_points[KEEP_21_INDICES]
    uv = project(selected)
    y3d = selected - selected[0:1]

    gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    del color

    affine = affine_for_crop(uv, IMAGE_W, IMAGE_H, TRAIN_CROP_MARGIN)
    crop_gray = cv2.warpAffine(
        gray, affine, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    crop_uv = uv @ affine[:, :2].T + affine[:, 2]

    records.append(
        {
            "segment": row["segment"],
            "frame_id": row.get("frame_id", str(i)),
            "image_path": image_path,
            "uv_image": uv.astype(np.float32),
            "uv_crop": crop_uv.astype(np.float32),
            "crop_gray": crop_gray,
            "y3d": y3d.astype(np.float32),
        }
    )

    if (i + 1) % 500 == 0 or i + 1 == len(rows):
        print(f"  staged {i + 1:,}/{len(rows):,} frames", flush=True)

y_all = np.stack([r["y3d"] for r in records]).astype(np.float32)
y2d_all = np.stack([normalize_uv(r["uv_crop"]) for r in records])
memory_status("after data staging")

# -------------------------- Sobel model -------------------------------------

def rotation_6d_to_matrix(d6):
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack((b1, b2, b3), dim=-1)


def reconstruct_joints(vectors):
    joints = torch.zeros(
        (vectors.shape[0], 21, 3), dtype=vectors.dtype, device=vectors.device
    )
    for k, (parent, child) in enumerate(BONE_PAIRS):
        joints[:, child] = joints[:, parent] + vectors[:, k]
    return joints


class SobelDirectTracker(nn.Module):
    def __init__(self, lengths):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8)),
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.meta_head = nn.Linear(512, 15)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 45)
        self.proj2d_head = nn.Linear(512, 42)
        self.register_buffer("bone_lens", lengths.clone())

    def forward(self, x):
        batch = x.shape[0]
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta = F.normalize(
            self.meta_head(feat).reshape(batch, 5, 3),
            dim=-1,
            eps=1e-7,
        )
        frame = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx = F.normalize(
            self.phalanx_head(feat).reshape(batch, 15, 3),
            dim=-1,
            eps=1e-7,
        )
        phalanx = torch.bmm(
            frame, phalanx.transpose(1, 2)
        ).transpose(1, 2)

        dirs = torch.zeros(
            (batch, 20, 3), dtype=x.dtype, device=x.device
        )
        dirs[:, 0] = meta[:, 0]
        dirs[:, 1:4] = phalanx[:, 0:3]
        for finger_i, offset in enumerate((4, 8, 12, 16)):
            start = 3 + finger_i * 3
            dirs[:, offset] = meta[:, finger_i + 1]
            dirs[:, offset + 1:offset + 4] = phalanx[
                :, start:start + 3
            ]

        joints = reconstruct_joints(
            dirs * self.bone_lens.view(1, 20, 1)
        )
        uv = torch.tanh(
            self.proj2d_head(feat).reshape(batch, 21, 2)
        )
        return joints, dirs, uv


def rotate_camera_xy(points, angle):
    c, s = np.cos(angle), np.sin(angle)
    out = points.copy()
    x, y = points[:, 0].copy(), points[:, 1].copy()
    out[:, 0] = c * x + s * y
    out[:, 1] = -s * x + c * y
    return out


def augment_sample(gray, uv, y3d, rng):
    angle_deg = float(rng.uniform(-15.0, 15.0))
    angle = np.deg2rad(angle_deg)
    scale = float(rng.uniform(0.90, 1.10))
    matrix = cv2.getRotationMatrix2D(
        (IMAGE_W * 0.5, IMAGE_H * 0.5), angle_deg, scale
    ).astype(np.float32)
    matrix[0, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_W)
    matrix[1, 2] += float(rng.uniform(-0.10, 0.10) * IMAGE_H)

    warped = cv2.warpAffine(
        gray, matrix, (IMAGE_W, IMAGE_H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    brightness = float(rng.uniform(0.80, 1.20))
    offset = float(rng.uniform(-16.0, 16.0))
    warped = np.clip(
        warped.astype(np.float32) * brightness + offset, 0, 255
    ).astype(np.uint8)
    uv_aug = uv @ matrix[:, :2].T + matrix[:, 2]
    return (
        sobel_channel(warped),
        rotate_camera_xy(y3d, angle).astype(np.float32),
        normalize_uv(uv_aug),
    )


def direct_loss(pred3d, target3d, pred2d, target2d):
    pos = F.l1_loss(pred3d, target3d)
    pred_bones = (
        pred3d[:, [c for _, c in BONE_PAIRS]]
        - pred3d[:, [p for p, _ in BONE_PAIRS]]
    )
    target_bones = (
        target3d[:, [c for _, c in BONE_PAIRS]]
        - target3d[:, [p for p, _ in BONE_PAIRS]]
    )
    pred_len = torch.linalg.vector_norm(pred_bones, dim=-1)
    target_len = torch.linalg.vector_norm(target_bones, dim=-1)
    length = F.l1_loss(pred_len, target_len)
    pred_unit = F.normalize(pred_bones, dim=-1, eps=1e-7)
    target_unit = F.normalize(target_bones, dim=-1, eps=1e-7)
    direction = (
        1.0 - torch.clamp((pred_unit * target_unit).sum(-1), -1.0, 1.0)
    ).mean()
    reprojection = F.smooth_l1_loss(pred2d, target2d)
    return pos + 0.5 * length + 20.0 * direction + 15.0 * reprojection


canonical_lengths = np.asarray(
    [
        np.linalg.norm(
            y_all[train_idx, child] - y_all[train_idx, parent],
            axis=-1,
        ).mean()
        for parent, child in BONE_PAIRS
    ],
    dtype=np.float32,
)
model = SobelDirectTracker(
    torch.from_numpy(canonical_lengths)
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-4
)
rng = np.random.default_rng(SEED + 1)
start_epoch = 0

# Resume automatically if a valid prior interrupted-run state is on Drive.
if DRIVE_AVAILABLE and RESUME_PATH.is_file():
    try:
        try:
            resume = torch.load(
                RESUME_PATH, map_location=DEVICE, weights_only=False
            )
        except TypeError:
            resume = torch.load(RESUME_PATH, map_location=DEVICE)

        same_run = (
            resume.get("seed") == SEED
            and resume.get("camera_id") == CAMERA_ID
            and resume.get("train_segments") == sorted(train_segments)
            and resume.get("val_segments") == sorted(val_segments)
        )
        if same_run:
            model.load_state_dict(resume["model_state_dict"])
            optimizer.load_state_dict(resume["optimizer_state_dict"])
            scheduler.load_state_dict(resume["scheduler_state_dict"])
            start_epoch = int(resume["completed_epoch"])
            print(
                f"Resuming Sobel training at epoch {start_epoch + 1}/"
                f"{N_EPOCHS}.",
                flush=True,
            )
        else:
            print(
                "Ignoring incompatible resume state; starting training fresh.",
                flush=True,
            )
    except Exception as exc:
        print(
            f"Could not load resume state ({exc}); starting fresh.",
            flush=True,
        )

# ----------------------------- Train with logs ------------------------------

print(
    f"Training Sobel direct model: {N_EPOCHS} epochs, "
    f"{len(train_idx):,} train frames, batch {BATCH_SIZE}.",
    flush=True,
)

for epoch in range(start_epoch, N_EPOCHS):
    epoch_start = time.perf_counter()
    model.train()
    order = np.random.permutation(train_idx)
    losses = []
    n_batches = (len(order) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_number, start in enumerate(
        range(0, len(order), BATCH_SIZE), start=1
    ):
        ids = order[start:start + BATCH_SIZE]
        samples = [
            augment_sample(
                records[int(i)]["crop_gray"],
                records[int(i)]["uv_crop"],
                y_all[int(i)],
                rng,
            )
            for i in ids
        ]
        bx = torch.from_numpy(np.stack([s[0] for s in samples])).to(DEVICE)
        by3d = torch.from_numpy(np.stack([s[1] for s in samples])).to(DEVICE)
        by2d = torch.from_numpy(np.stack([s[2] for s in samples])).to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred3d, _, pred2d = model(bx)
        loss = direct_loss(pred3d, by3d, pred2d, by2d)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

        # Heartbeat output prevents long silent intervals on slow CPUs.
        if batch_number % 25 == 0 or batch_number == n_batches:
            print(
                f"  epoch {epoch + 1:02d}/{N_EPOCHS}, "
                f"batch {batch_number}/{n_batches}, "
                f"loss={np.mean(losses):.3f}",
                flush=True,
            )

    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    print(
        f"Epoch {epoch + 1:02d}/{N_EPOCHS} complete; "
        f"mean loss={np.mean(losses):.4f}; "
        f"{time.perf_counter() - epoch_start:.1f}s",
        flush=True,
    )

    # Save resume state after each epoch. An interrupted run can resume.
    if DRIVE_AVAILABLE:
        temporary_path = RESUME_PATH.with_suffix(".tmp")
        state = {
            "model_state_dict": {
                k: v.detach().cpu() for k, v in model.state_dict().items()
            },
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "completed_epoch": epoch + 1,
            "seed": SEED,
            "camera_id": CAMERA_ID,
            "train_segments": sorted(train_segments),
            "val_segments": sorted(val_segments),
        }
        torch.save(state, temporary_path)
        os.replace(temporary_path, RESUME_PATH)
        print("  resumable state saved to Drive", flush=True)

model.eval()
memory_status("after training")

# Save an inference-ready checkpoint to Drive (or local /content fallback).
final_checkpoint = {
    "model_state_dict": {
        k: v.detach().cpu() for k, v in model.state_dict().items()
    },
    "canonical_bone_lengths_mm": canonical_lengths,
    "seed": SEED,
    "train_segments": sorted(train_segments),
    "val_segments": sorted(val_segments),
    "camera_id": CAMERA_ID,
    "manifest_path": str(MANIFEST_PATH),
}
if DRIVE_AVAILABLE:
    FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(final_checkpoint, FINAL_MODEL_PATH)
    saved_checkpoint_path = FINAL_MODEL_PATH
else:
    saved_checkpoint_path = Path("/content/phase3_sobel_direct.pt")
    torch.save(final_checkpoint, saved_checkpoint_path)

# ------------------------- MediaPipe candidate pass -------------------------

mp_options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(MP_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
)


def result_candidates(result, size, gt_uv, mirrored):
    candidates = []
    image_hands = getattr(result, "hand_landmarks", None) or []
    world_hands = getattr(result, "hand_world_landmarks", None) or []
    handedness = getattr(result, "handedness", None) or []

    for hand_i, (image_hand, world_hand) in enumerate(
        zip(image_hands, world_hands)
    ):
        if len(image_hand) != 21 or len(world_hand) != 21:
            continue

        xy = np.asarray(
            [[lm.x * size, lm.y * size] for lm in image_hand],
            dtype=np.float32,
        )
        if mirrored:
            xy[:, 0] = size - 1 - xy[:, 0]

        pixel_error = float(np.linalg.norm(xy - gt_uv, axis=-1).mean())
        world_m = np.asarray(
            [[lm.x, lm.y, lm.z] for lm in world_hand],
            dtype=np.float32,
        )
        pose = (world_m - world_m[0:1]) * 1000.0
        if mirrored:
            pose[:, 0] *= -1.0

        label = "unknown"
        if hand_i < len(handedness):
            try:
                label = handedness[hand_i][0].category_name
            except (AttributeError, IndexError, TypeError):
                pass

        candidates.append(
            {
                "pose": pose.astype(np.float32),
                "xy": xy,
                "pixel_error": pixel_error,
                "mirrored_input": bool(mirrored),
                "hand_index": hand_i,
                "handedness": label,
            }
        )
    return candidates


all_candidates = [[] for _ in val_idx]
normal_2d_errors, best_2d_errors = [], []
val_pos_by_dataset_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

print(
    f"Running MediaPipe on {len(val_idx):,} validation frames "
    "(normal + mirrored color crops)...",
    flush=True,
)

with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for val_pos, dataset_i in enumerate(val_idx):
        record = records[int(dataset_i)]
        color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
        if color is None:
            raise FileNotFoundError(record["image_path"])

        crop, gt_crop_uv = make_mp_crop(color, record["uv_image"])
        crop_size = crop.shape[0]
        candidates = []

        for mirrored in (False, True):
            view = cv2.flip(crop, 1) if mirrored else crop
            rgb = cv2.cvtColor(view, cv2.COLOR_BGR2RGB)
            result = landmarker.detect(
                mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(rgb),
                )
            )
            candidates.extend(
                result_candidates(
                    result, crop_size, gt_crop_uv, mirrored
                )
            )

        all_candidates[val_pos] = candidates
        normal = [c for c in candidates if not c["mirrored_input"]]
        if normal:
            normal_2d_errors.append(
                min(normal, key=lambda c: c["pixel_error"])["pixel_error"]
            )
        if candidates:
            best_2d_errors.append(
                min(candidates, key=lambda c: c["pixel_error"])["pixel_error"]
            )

        if (val_pos + 1) % 50 == 0 or val_pos + 1 == len(val_idx):
            print(
                f"  MediaPipe {val_pos + 1}/{len(val_idx)} frames",
                flush=True,
            )

detected_positions = np.asarray(
    [i for i, candidates in enumerate(all_candidates) if candidates],
    dtype=np.int64,
)
if not len(detected_positions):
    raise RuntimeError("MediaPipe found no hands on validation frames.")

nearest_candidates = [
    min(c, key=lambda item: item["pixel_error"]) if c else None
    for c in all_candidates
]

# ------------------------- Batched Sobel inference ---------------------------

sobel_predictions = [None] * len(val_idx)
print("Running batched Sobel inference...", flush=True)

with torch.inference_mode():
    for start in range(0, len(val_idx), 128):
        pos_batch = list(range(start, min(start + 128, len(val_idx))))
        x_np = np.stack([
            sobel_channel(records[int(val_idx[pos])]["crop_gray"])
            for pos in pos_batch
        ])
        x = torch.from_numpy(x_np).to(DEVICE)
        joints, _, _ = model(x)
        pred = joints.cpu().numpy().astype(np.float32)
        for j, pos in enumerate(pos_batch):
            sobel_predictions[pos] = pred[j]

gt_val = [records[int(i)]["y3d"] for i in val_idx]


def mpjpe(pred, target):
    return float(np.linalg.norm(pred - target, axis=-1).mean())


def rescale_bones(pose):
    result = np.zeros_like(pose, dtype=np.float32)
    for bone_i, (parent, child) in enumerate(BONE_PAIRS):
        direction = pose[child] - pose[parent]
        norm = float(np.linalg.norm(direction))
        if norm < 1e-8:
            direction = np.asarray([1.0, 0.0, 0.0], dtype=np.float32)
        else:
            direction = direction / norm
        result[child] = result[parent] + direction * canonical_lengths[bone_i]
    return result


def pa_mpjpe_one(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    x = pred - pred.mean(axis=0, keepdims=True)
    y = target - target.mean(axis=0, keepdims=True)

    u, singular, vt = np.linalg.svd(x.T @ y)
    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(u @ vt)
    rotation = u @ correction @ vt
    denominator = np.sum(x * x)
    if denominator < 1e-12:
        return float("nan")
    scale = np.sum(singular * np.diag(correction)) / denominator
    aligned = scale * (x @ rotation) + target.mean(axis=0, keepdims=True)
    return float(np.linalg.norm(aligned - target, axis=-1).mean())


mp_raw = [None] * len(val_idx)
mp_scaled = [None] * len(val_idx)
for pos in detected_positions:
    pose = nearest_candidates[int(pos)]["pose"]
    mp_raw[int(pos)] = pose
    mp_scaled[int(pos)] = rescale_bones(pose)

# ---------------------------- Audit tables ----------------------------------

print("\nFRAME AUDIT — 2D-NEAREST CANDIDATE")
print(f"{'XYZ signs':<12} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 34)
for signs in itertools.product((1.0, -1.0), repeat=3):
    errors = [
        mpjpe(
            nearest_candidates[int(pos)]["pose"]
            * np.asarray(signs, dtype=np.float32),
            gt_val[int(pos)],
        )
        for pos in detected_positions
    ]
    label = "".join("+" if s > 0 else "-" for s in signs)
    print(f"{label:<12} {len(errors):>8} {np.mean(errors):10.2f}")

print("\n2D CHECK — MEAN LANDMARK DISTANCE")
print(f"{'Candidate pool':<36} {'Frames':>8} {'Error px':>10}")
print("-" * 58)
print(
    f"{'Original crop only':<36} {len(normal_2d_errors):>8} "
    f"{np.mean(normal_2d_errors):10.2f}"
)
print(
    f"{'Best original/mirrored per frame':<36} {len(best_2d_errors):>8} "
    f"{np.mean(best_2d_errors):10.2f}"
)

nearest_errors, oracle_errors = [], []
different_candidates = 0
for pos in detected_positions:
    pos = int(pos)
    candidates = all_candidates[pos]
    nearest = min(candidates, key=lambda c: c["pixel_error"])
    oracle = min(candidates, key=lambda c: mpjpe(c["pose"], gt_val[pos]))
    nearest_errors.append(mpjpe(nearest["pose"], gt_val[pos]))
    oracle_errors.append(mpjpe(oracle["pose"], gt_val[pos]))
    different_candidates += int(
        (nearest["mirrored_input"], nearest["hand_index"])
        != (oracle["mirrored_input"], oracle["hand_index"])
    )

print("\nCANDIDATE CHECK")
print(f"{'Selection':<28} {'Frames':>8} {'MPJPE mm':>10}")
print("-" * 50)
print(
    f"{'2D-nearest':<28} {len(nearest_errors):>8} "
    f"{np.mean(nearest_errors):10.2f}"
)
print(
    f"{'3D oracle':<28} {len(oracle_errors):>8} "
    f"{np.mean(oracle_errors):10.2f}"
)
print(
    f"Different candidate identity: {different_candidates}/"
    f"{len(detected_positions)} "
    f"({different_candidates / len(detected_positions):.2%})"
)

print("\nPA-MPJPE — PER-FRAME ROTATION + SCALE")
print(f"{'Method':<26} {'Frames':>8} {'PA-MPJPE mm':>13}")
print("-" * 52)
for name, predictions in (
    ("MediaPipe raw", mp_raw),
    ("MediaPipe bone-scaled", mp_scaled),
    ("Sobel direct", sobel_predictions),
):
    errors = [
        pa_mpjpe_one(predictions[int(pos)], gt_val[int(pos)])
        for pos in detected_positions
    ]
    print(
        f"{name:<26} {len(errors):>8} "
        f"{np.nanmean(errors):13.2f}"
    )

# ------------------------- Detection by segment ----------------------------

segment_stats = {}
for val_pos, dataset_i in enumerate(val_idx):
    segment = records[int(dataset_i)]["segment"]
    entry = segment_stats.setdefault(segment, {"frames": 0, "detected": 0})
    entry["frames"] += 1
    entry["detected"] += int(bool(all_candidates[val_pos]))

print("\nDETECTION BY VALIDATION SEGMENT")
print(f"{'Segment':<34} {'Detected':>9} {'Frames':>8} {'Rate':>9}")
print("-" * 64)
for segment in sorted(segment_stats, key=str):
    entry = segment_stats[segment]
    rate = entry["detected"] / max(entry["frames"], 1)
    print(
        f"{str(segment):<34} {entry['detected']:>9} "
        f"{entry['frames']:>8} {rate:>8.2%}"
    )

# -------------------- Four overlays across all records ----------------------

def draw_overlay(candidate, crop, gt_uv, path):
    canvas = crop.copy()
    gt_pts = np.rint(gt_uv).astype(int)
    mp_pts = np.rint(candidate["xy"]).astype(int)

    for parent, child in BONE_PAIRS:
        cv2.line(canvas, tuple(gt_pts[parent]), tuple(gt_pts[child]),
                 (0, 255, 0), 2, cv2.LINE_AA)
        cv2.line(canvas, tuple(mp_pts[parent]), tuple(mp_pts[child]),
                 (0, 0, 255), 2, cv2.LINE_AA)
    for point in gt_pts:
        cv2.circle(canvas, tuple(point), 4, (0, 255, 0), -1, cv2.LINE_AA)
    for point in mp_pts:
        cv2.circle(canvas, tuple(point), 4, (255, 0, 0), -1, cv2.LINE_AA)

    cv2.putText(
        canvas,
        f"GT green; MP red/blue; mirrored={candidate['mirrored_input']}",
        (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
        (255, 255, 255), 1, cv2.LINE_AA,
    )
    if not cv2.imwrite(str(path), canvas):
        raise IOError(f"Could not save overlay {path}")


OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
overlay_paths = []
print("\nSaving overlays for T0011 and T0020...", flush=True)

# A constant-time map avoids scanning all validation indices for every record.
val_position_by_index = {
    int(dataset_i): pos for pos, dataset_i in enumerate(val_idx)
}

# Keep one detector open; use cached val candidates where possible, and run
# detection only for requested target-segment frames that were in train.
with mp.tasks.vision.HandLandmarker.create_from_options(mp_options) as landmarker:
    for segment_name in ("T0011", "T0020"):
        saved_count = 0

        for record_i, record in enumerate(records):
            if str(record["segment"]) != segment_name:
                continue

            color = cv2.imread(str(record["image_path"]), cv2.IMREAD_COLOR)
            if color is None:
                continue
            crop, gt_crop_uv = make_mp_crop(color, record["uv_image"])

            val_pos = val_position_by_index.get(record_i)
            if val_pos is not None and all_candidates[val_pos]:
                candidate = min(
                    all_candidates[val_pos],
                    key=lambda c: c["pixel_error"],
                )
            else:
                size = crop.shape[0]
                found = []
                for mirrored in (False, True):
                    view = cv2.flip(crop, 1) if mirrored else crop
                    rgb = cv2.cvtColor(view, cv2.COLOR_BGR2RGB)
                    result = landmarker.detect(
                        mp.Image(
                            image_format=mp.ImageFormat.SRGB,
                            data=np.ascontiguousarray(rgb),
                        )
                    )
                    found.extend(
                        result_candidates(
                            result, size, gt_crop_uv, mirrored
                        )
                    )
                if not found:
                    continue
                candidate = min(found, key=lambda c: c["pixel_error"])

            saved_count += 1
            frame_id = str(record.get("frame_id", record_i))
            safe_id = "".join(
                c if c.isalnum() or c in "-_" else "_"
                for c in frame_id
            )
            path = OVERLAY_DIR / f"{segment_name}_{saved_count}_{safe_id}.png"
            draw_overlay(candidate, crop, gt_crop_uv, path)
            overlay_paths.append(path)

            if saved_count == 2:
                break

        if saved_count < 2:
            print(
                f"Warning: only {saved_count} detected overlay frame(s) "
                f"found for {segment_name}.",
                flush=True,
            )

print("\nOVERLAYS")
for path in overlay_paths:
    print(path)

memory_status("finished")
print(f"Saved model checkpoint: {saved_checkpoint_path}")
print(
    "Assumptions: right-hand GT is orig_kp_right; source 3D points are mm; "
    "MediaPipe world coordinates are meters and are re-rooted at wrist 0. "
    "Mirrored-input 3D x is reflected before scoring. The 3D oracle uses GT "
    "and is an upper-bound diagnostic. PA-MPJPE uses a proper rotation and "
    "one uniform scale per frame. T0011/T0020 overlays search all records, "
    "not only validation records."
)